In [10]:
import torch
# Enable TF32 for Matrix Multiplication (matmul)
torch.backends.cuda.matmul.allow_tf32 = True

# Enable TF32 for Convolutions (cudnn)
torch.backends.cudnn.allow_tf32 = True


In [ ]:
from config.CustomCLMConfig import NoraConfig
from config.ModelSettings import CMSConfig
from transformers import AutoModelForCausalLM
import torch.nn as nn
from transformers import AutoConfig, AutoModelForCausalLM
# from config.ModelSettings import NoraConfig
from model.CausalLM import NoraCausalLM
import torch

checkpoint_path = "/root/KartikGoyal/nora_experiment_v5/checkpoints/checkpoint_epoch_15.pt"
cms_config = CMSConfig(
        3072,           # hidden dim
        7,              # num blocks
        [1, 2, 4, 8, 16, 32, 64],   # frequencies
        [6, 4, 4, 3, 3, 2, 2],      # expansion ratios
        nn.GELU,)
    
config = NoraConfig(model_name="llama-3.2-3b", cms_cfg=cms_config)
print("\n\n",config.dtype, "\n\n")

AutoConfig.register("NORA", NoraConfig)
AutoModelForCausalLM.register(NoraConfig, NoraCausalLM)
checkpoint = torch.load(
            checkpoint_path, weights_only=False, map_location="cuda:1"
    )
model = AutoModelForCausalLM.from_config(config)
model = torch.compile(model)
model.load_state_dict(checkpoint['model_state_dict'])
# ---- Save & reload ----
# model.save_pretrained("./nora-checkpoint")

# model = AutoModelForCausalLM.from_pretrained("./nora-checkpoint")

# ---- Generation (beam search, sampling, etc. — all work) ----
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-3B")
inputs = tokenizer("Hello, I am Nora", return_tensors="pt")
print(tokenizer.name_or_path)

# Check if '下' is even in the vocabulary
ids = tokenizer.encode("下", add_special_tokens=False)
print(f"'下' encodes to: {ids}")
print(f"Those IDs decode to: {[tokenizer.decode([i]) for i in ids]}")

# output = model.generate(**inputs, max_new_tokens=100, do_sample=True)

# print(tokenizer.decode(output[0]))

# Check if LM head output dim matches tokenizer vocab
print("LM head output dim:", model.nora.lm_head.weight.shape[0])
print("Tokenizer vocab size:", tokenizer.vocab_size)
print("Tokenizer total vocab:", len(tokenizer))


/root/KartikGoyal/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm




 None 




Loading weights: 100%|██████████| 254/254 [00:00<00:00, 5826.98it/s]


KeyboardInterrupt: 

In [6]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-3B")

In [ ]:
print("LM head output dim:", model.nora.lm_head.weight.shape[0])
print("Tokenizer vocab size:", tokenizer.vocab_size)
print("Tokenizer total vocab:", len(tokenizer))


LM head output dim: 128256
Tokenizer vocab size: 128000
Tokenizer total vocab: 128256


In [12]:
import torch
from torch.amp import autocast

device = torch.device("cuda:1")
model.to(device)
model.eval()

def sample_next_token(logits_bf16, temperature=0., top_k=50, top_p=0.9):
    # Cast to float32 — critical fix
    logits = logits_bf16.float()
    
    # Temperature
    logits = logits / temperature
    
    # Top-k
    top_k_vals = torch.topk(logits, top_k).values
    logits[logits < top_k_vals[-1]] = float('-inf')
    
    # Top-p
    sorted_logits, sorted_idx = torch.sort(logits, descending=True)
    cumprobs = torch.cumsum(torch.softmax(sorted_logits, dim=-1), dim=-1)
    sorted_logits[cumprobs > top_p] = float('-inf')
    logits[sorted_idx] = sorted_logits
    
    probs = torch.softmax(logits, dim=-1)
    return probs, torch.multinomial(probs, num_samples=1)
with torch.no_grad():
    text = """The French Revolution was a period of far-reaching social and political upheaval in France that began in 1789 and lasted until"""
    enc = tokenizer(text, return_tensors="pt").to(device)
    
    input_ids = enc["input_ids"]
    
    def sample_token(model, input_ids, temperature=1.5, top_k=50, top_p=0.9):
        with autocast("cuda", dtype=torch.bfloat16):
            out = model(input_ids=input_ids)
        
        logits = out.logits[0, -1, :].float()
        logits = logits / temperature
        
        # Top-k
        filtered_logits = torch.full_like(logits, float('-inf'))
        top_k_vals, top_k_idx = torch.topk(logits, top_k)
        filtered_logits[top_k_idx] = top_k_vals
        
        # Top-p on top of top-k
        sorted_logits, sorted_idx = torch.sort(filtered_logits, descending=True)
        sorted_probs = torch.softmax(sorted_logits, dim=-1)
        cumprobs = torch.cumsum(sorted_probs, dim=-1)
        remove_mask = cumprobs - sorted_probs > top_p
        sorted_logits[remove_mask] = float('-inf')
        filtered_logits[sorted_idx] = sorted_logits
        
        probs = torch.softmax(filtered_logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        
        # Show distribution for this step
        top_probs, top_ids = probs.topk(5)
        return next_id, top_probs, top_ids

    print(f"Prompt: '{text}'\n")
    print(f"{'Step':<6} {'Sampled':<20} {'Top 5 candidates'}")
    print("-" * 80)
    
    for step in range(400):
        next_id, top_probs, top_ids = sample_token(model, input_ids)
        
        candidates = "  |  ".join([
            f"{repr(tokenizer.decode([tid.item()]))}: {prob.item()*100:.2f}%"
            for tid, prob in zip(top_ids, top_probs)
        ])
        
        sampled_token = repr(tokenizer.decode([next_id.item()]))
        print(f"  {step+1:<4} {sampled_token:<20} {candidates}")
        
        input_ids = torch.cat([input_ids, next_id.unsqueeze(0)], dim=-1)
    
    final_text = tokenizer.decode(input_ids[0], skip_special_tokens=True)
    print(f"\nFull output: '{final_text}'")
    print(f"\n'下' generated: {'yes' if 17297 in input_ids[0].tolist() else 'no'}")

Prompt: 'The French Revolution was a period of far-reaching social and political upheaval in France that began in 1789 and lasted until'

Step   Sampled              Top 5 candidates
--------------------------------------------------------------------------------
  1    ' the'               ' ': 48.39%  |  ' the': 40.96%  |  ' about': 5.54%  |  ' around': 5.10%  |  '!': 0.00%
  2    ' fall'              ' fall': 49.32%  |  ' French': 19.72%  |  ' establishment': 18.14%  |  ' end': 6.67%  |  ' overthrow': 6.14%
  3    ' of'                ' of': 100.00%  |  '#': 0.00%  |  '!': 0.00%  |  '$': 0.00%  |  '"': 0.00%
  4    ' Napoleon'          ' Napoleon': 54.16%  |  ' the': 45.84%  |  '#': 0.00%  |  '"': 0.00%  |  '!': 0.00%
  5    ' in'                ' Bon': 63.22%  |  ' in': 27.48%  |  ' I': 9.30%  |  '"': 0.00%  |  '!': 0.00%
  6    ' '                  ' ': 100.00%  |  '#': 0.00%  |  '!': 0.00%  |  '$': 0.00%  |  '"': 0.00%
  7    '181'                '181': 100.00%  |  '#': 0.00%  | 

In [ ]:
# Run a single greedy forward pass and check raw logits
model.eval()
with torch.no_grad():
    enc = tokenizer("The history of", return_tensors="pt").to(device)
    out = model(**enc)
    logits = out.logits[0, -1, :]  # last token logits
    top_ids = logits.topk(10).indices.tolist()
    print("Top 10 predicted next tokens:")
    for tid in top_ids:
        print(f"  id={tid:6d}  '{tokenizer.decode([tid])}'  logit={logits[tid]:.3f}")